In [ ]:
import logging

from library.circuitry import Circuitry
from library.junction_patch import JunctionPatch
from library.qubit_array import QubitArray
from library.surface_code.expanding_patch import ExpandingSurfaceCodePatch
from library.surface_code.patch import SurfaceCodePatch, PauliBasis
from library.steane_code.patch import SteaneCodePatch

logging.basicConfig(level=logging.ERROR)
from IPython.display import Markdown
from utils.error_rate_analyser import simulate

In [ ]:
def detector_report(circuitry: Circuitry) -> str:
    if not circuitry.is_clifford:
        return "n/a [non-Clifford circuitry]"

    issues = []
    if len(circuitry.as_stim.missing_detectors()) > 0:
        issues.append("MISSING")

    try:
        circuitry.as_stim.detector_error_model(allow_gauge_detectors=False)
    except ValueError:
        issues.append("GAUGE")

    return "&".join(issues)

In [ ]:
INJECTION = SteaneCodePatch.Injection.S
TARGET_DISTANCE = 9

SUPERDENSE_ROUNDS = 3
TELEPORT_ROUNDS = 3
ROUNDS_FOR_COMPLEMENTARY_GAP = 1

if TARGET_DISTANCE % 2 != 1 and TARGET_DISTANCE < 9:
    raise ValueError("TARGET_DISTANCE must be odd and above 9.")

In [ ]:
FILEROOT = "generated/hirano-magic-state-cultivation-layout1"
scenarios: dict[str, Circuitry] = dict()
point = 0

In [ ]:
# Generate circuit up to and including preparation (w/ S-injection)
circuitry = Circuitry(clifford=INJECTION == SteaneCodePatch.Injection.S)
array = QubitArray(circuitry, dimensions=(TARGET_DISTANCE + 2, TARGET_DISTANCE + 2))
steane = SteaneCodePatch(array, anchor=(TARGET_DISTANCE-5, TARGET_DISTANCE-7), injection=INJECTION)

# Append preparation
steane.append_preparation(circuitry)
steane.append_observable(circuitry, observable='Y', support = steane.logical)
scenarios[f'Point {point} - Prepared'] = circuitry
circuitry.to_file(FILEROOT + f".point{point}.prepared")
point += 1

In [ ]:
# Generate circuit up to and including superdense code cycles
circuitry = Circuitry(clifford=INJECTION == SteaneCodePatch.Injection.S)
array = QubitArray(circuitry, dimensions=(TARGET_DISTANCE + 2, TARGET_DISTANCE + 2))
steane = SteaneCodePatch(array, anchor=(TARGET_DISTANCE-5, TARGET_DISTANCE-7), injection=INJECTION)

steane.append_preparation(circuitry)
for s in range(SUPERDENSE_ROUNDS):
    label = f"SDC{s}"
    steane.append_superdense_cycle(circuitry, prefix=label)

steane.annotate_detectors(circuitry, sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=0)
steane.append_observable(circuitry, observable='Y', support = steane.logical)

scenarios[f'Point {point} - SDCx{SUPERDENSE_ROUNDS}'] = circuitry
circuitry.to_file(FILEROOT + f".point{point}.superdense")
point += 1

In [ ]:
# Generate circuit up to and including double-check-S
circuitry = Circuitry(clifford=INJECTION == SteaneCodePatch.Injection.S)
array = QubitArray(circuitry, dimensions=(TARGET_DISTANCE + 2, TARGET_DISTANCE + 2))
steane = SteaneCodePatch(array, anchor=(TARGET_DISTANCE-5, TARGET_DISTANCE-7), injection=INJECTION)

steane.append_preparation(circuitry)
for s in range(SUPERDENSE_ROUNDS):
    label = f"SDC{s}"
    steane.append_superdense_cycle(circuitry, prefix=label)
steane.append_cultivation(circuitry, prefix="CULT")

steane.annotate_detectors(circuitry, sdc_rounds=SUPERDENSE_ROUNDS)
steane.append_observable(circuitry, observable='Y', support = steane.logical)

scenarios[f'Point {point} - Double-Check-S'] = circuitry
circuitry.to_file(FILEROOT + f".point{point}.double-check-s")
point += 1

In [ ]:
# Generate circuit up to and including teleportation
circuitry = Circuitry(clifford=INJECTION == SteaneCodePatch.Injection.S)
array = QubitArray(circuitry, dimensions=(TARGET_DISTANCE + 2, TARGET_DISTANCE + 2))
steane = SteaneCodePatch(array, anchor=(TARGET_DISTANCE-5, TARGET_DISTANCE-7), injection=INJECTION)
junction = JunctionPatch(array, anchor=(TARGET_DISTANCE-5, TARGET_DISTANCE-5))
source = SurfaceCodePatch(array, distance=5, anchor=(TARGET_DISTANCE - 4, TARGET_DISTANCE - 4))
inactive_source = lambda location: location[1] == TARGET_DISTANCE - 4.5

steane.append_preparation(circuitry)
for s in range(SUPERDENSE_ROUNDS):
    label = f"SDC{s}"
    steane.append_superdense_cycle(circuitry, prefix=label)
steane.append_cultivation(circuitry, prefix="CULT")
for rnd in range(TELEPORT_ROUNDS):
    for mmt in steane.TELEPORTATION_MOMENTS:
        steane.append_teleportation_slice(circuitry, moment=mmt, prefix=f"TPT{rnd}")
        junction.append_syndrome_slice(circuitry, moment=mmt, prefix=f"JCT{rnd}")
        source.append_round_slice(
            circuitry, moment=mmt, prepare=PauliBasis.X if rnd == 0 else None, prefix=f"SC{rnd}",
            inactive = inactive_source
        )
        circuitry.append_tick()
steane.append_destruction(circuitry)
source.append_round(circuitry, prefix=f"SC{TELEPORT_ROUNDS}")

source.append_general_observable(circuitry, {
    (0,0) : "Y", (1,0) : "Z", (2,0) : "Z", (3,0) : "Z", (4,0) : "Z", (0,1) : "X", (0,2) : "X", (0,3) : "X", (0,4) : "X",
}, "JCT0:Z0", "JCT0:Z1", "JCT0:Z2", "TPT0:XB", "TPT1:XB", "TPT2:XB", "DST:X1", "DST:X5", "DST:X6")

steane.annotate_detectors(circuitry, sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=TELEPORT_ROUNDS)
junction.annotate_detectors(circuitry, rounds=TELEPORT_ROUNDS)
source.annotate_detectors(circuitry, rounds=TELEPORT_ROUNDS + 1, prepared=PauliBasis.X)

scenarios[f'Point {point} - Teleported'] = circuitry
circuitry.to_file(FILEROOT + f".point{point}.teleported")
point += 1

In [ ]:
# Generate the circuit up to and including expansion :)
circuitry = Circuitry(clifford=INJECTION == SteaneCodePatch.Injection.S)
array = QubitArray(circuitry, dimensions=(TARGET_DISTANCE + 2, TARGET_DISTANCE + 2))
steane = SteaneCodePatch(array, anchor=(TARGET_DISTANCE-5, TARGET_DISTANCE-7), injection=INJECTION)
junction = JunctionPatch(array, anchor=(TARGET_DISTANCE-5, TARGET_DISTANCE-5))
source = SurfaceCodePatch(array, distance=5, anchor=(TARGET_DISTANCE - 4, TARGET_DISTANCE - 4))
inactive_source = lambda location: location[1] == TARGET_DISTANCE - 4.5
expanding = ExpandingSurfaceCodePatch(array, distance=5, anchor=(1, 1), expansion=TARGET_DISTANCE - 5)

steane.append_preparation(circuitry)
for s in range(SUPERDENSE_ROUNDS):
    label = f"SDC{s}"
    steane.append_superdense_cycle(circuitry, prefix=label)
steane.append_cultivation(circuitry, prefix="CULT")
for rnd in range(TELEPORT_ROUNDS):
    for mmt in steane.TELEPORTATION_MOMENTS:
        steane.append_teleportation_slice(circuitry, moment=mmt, prefix=f"TPT{rnd}")
        junction.append_syndrome_slice(circuitry, moment=mmt, prefix=f"JCT{rnd}")
        source.append_round_slice(
            circuitry, moment=mmt, prepare=PauliBasis.X if rnd == 0 else None, prefix=f"SC{rnd}",
            inactive = inactive_source
        )
        circuitry.append_tick()
steane.append_destruction(circuitry)
source.append_round(circuitry, prefix=f"SC{TELEPORT_ROUNDS}")
for mmt in expanding.MOMENTS:
    expanding.append_expansion_slice(circuitry, moment=mmt, prefix=f"EXP")
    circuitry.append_tick()

cross = TARGET_DISTANCE-5
logical_observable = { (cross, cross) : "Y" }
for i in range(TARGET_DISTANCE):
    if i == cross:
        continue
    logical_observable[(i,cross)] = "Z"
    logical_observable[(cross,i)] = "X"
expanding.append_general_observable(
    circuitry, logical_observable, "JCT0:Z0", "JCT0:Z1", "JCT0:Z2", "TPT0:XB", "TPT1:XB", "TPT2:XB", "DST:X1", "DST:X5", "DST:X6"
)

steane.annotate_detectors(circuitry, sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=TELEPORT_ROUNDS)
junction.annotate_detectors(circuitry, rounds=TELEPORT_ROUNDS)
source.annotate_detectors(circuitry, rounds=TELEPORT_ROUNDS + 1, prepared=PauliBasis.X)
expanding.annotate_detectors(circuitry, sc_rounds=TELEPORT_ROUNDS + 1, source=source)

scenarios[f'Point {point} - Expanded'] = circuitry
circuitry.to_file(FILEROOT + f".point{point}.expanded")
point += 1

In [ ]:
for point, circuitry in scenarios.items():
    warning = detector_report(circuitry)
    if circuitry.is_clifford:
        display(Markdown(f"[Open in Crumble ({point})]({circuitry.as_stim.to_crumble_url()}) {warning}"))

In [ ]:
# Analyse error rates of all cumulative circuits
title = r"Magic State Cultivation of $|\mathbf{S}\rangle$ [$\mathbf{Y}$-observables]"
simulate(scenarios, title, postselection=True, shots=1e5, minimal_noise=-5, figsize=(11, 4.5), num_workers=7, filename=FILEROOT)